## 1. Motivation and model overview {#motivation-and-model-overview}

At each moment of individual ball control, a player either releases a pass or keeps carrying the ball. Action selection estimates the probability of these two observed decisions from the current tracking state and match context. Stalpaert et al. [@stalpaert2022] study the same binary pass–carry choice, while Fernández et al. [@fernandez2021] treat a related problem with three actions (shooting, passing, carrying) inside a possession-value framework. PRESS covers outfield players and the pass–carry decision; shooting is not modelled.

Let $s$ denote the current game state and $A$ the chosen action. The target is

$$
q(s)=P(A=\mathrm{pass}\mid s),\qquad
P(A=\mathrm{carry}\mid s)=1-q(s).
$$

We model action selection as a **binary classification problem** and estimate $q(s)$ with an XGBoost classifier [@chen2016xgboost]. The classifier takes a vector of features that summarizes the game state and returns the probability of passing. The features are described first, followed by the likelihood used to train the model.

These are probabilities of *choosing* an action, and the model describes observed behaviour across the sampled decisions. Before choosing, a player weighs the risk of carrying, the space available and the prospects of a pass. PRESS represents these considerations through features computed from tracking data and from its separately fitted execution and destination-choice models.

## 2. Model features {#model-features}

The model uses 30 features, 28 numerical and two categorical, in four groups. Each group answers one question about the decision. Each feature is described first in plain terms and then by how it is computed.

| Group | Question | Count |
|---|---|---:|
| Carrier state and ball retention | How well can the player keep carrying? | 6 |
| Spatial and tactical configuration | Where does the carry lead, and how are the teams organized there? | 8 |
| Passing opportunities | What would a pass look like, and how would it end? | 11 |
| Possession and match context | How did control start, and what is the match situation? | 5 |

### Carrier state and ball retention (6 features)

These features describe the player and the room available to keep moving with the ball.

- **x, y position.** Where the carrier is on the pitch, in the team's attacking orientation: deep in the team's own half or near the opponent's goal, central or wide.
- **Speed.** How fast the carrier is moving: running with the ball, or almost standing.
- **Velocity angle.** The direction of movement relative to the attacking axis: forward, sideways or backward. It is missing when the player is effectively stationary. Velocity comes from the centred seven-frame tracking smoother, whose three-frame future support makes the current implementation suitable for offline analysis.
- **Carry-success probability $1-d(s)$.** The chance that the player still has the ball one second from now if carrying continues. A high value means carrying is safe in the current situation. It comes from the separate [ball-drive risk model](../01_ball_drive_risk/01_ball_drive_risk.ipynb) and measures execution risk.
- **BCOS area.** The size of the space the carrier can reach before any opponent, in $\mathrm{m}^2$. A large value means room to run, a small one means the carrier is boxed in. Ball carrier open space (BCOS) is a concept introduced for association football by Bandara et al. [@bandara2025bcos]. In PRESS, the [BCOS model](../spatial_features/bcos/10_bcos.ipynb) gives, for each point around the carrier, the probability that the carrier arrives there before an opponent can intercept; integrating this field over a 40 m surrounding domain gives the probability-weighted area.

The carry-loss probability is defined as

$$
d(s)=P(\mathrm{loss\ within\ 1\,s}\mid A=\mathrm{carry},s),
$$

and the feature $1-d(s)$ tells whether control would survive the next second, and BCOS area tells how much space is available. Both describe the carry option from the same current state.

### Spatial and tactical configuration (8 features)

These features describe where the carry would take the player and how the two teams are arranged around that place.

Six of them are evaluated at a **projected position**: where the carrier would be one second from now if the carrier kept the current velocity,

$$
\widehat x_{\mathrm{bc}}(1)=x_{\mathrm{bc}}(0)+v_{\mathrm{bc}}(0)\,(1\,\mathrm{s}).
$$

This is a hypothetical point: the features ask what the situation looks like where the carry is heading.

- **Own-team xT.** The expected threat of the projected position for the carrier's team, that is, how much a possession at that spot tends to lead towards a goal. A high value means the carry heads into a dangerous attacking zone. xT was introduced by Singh [@singh2018xt]; its construction in PRESS is described in the [xT vignette](../spatial_features/expected_threat/11_expected_threat.ipynb).
- **Opponent xT.** The same quantity from the opponent's point of view. It indicates how dangerous that spot would be for the carrier's team if possession were lost there.
- **Nearest own-team line index** and **nearest opposition line index.** These are the pressure lines of Fernández, Bornn and Cervone [@fernandez2021]. In PRESS, the current outfield longitudinal positions are clustered into three lines with complete linkage, excluding the original carrier from the own-team roster, padding and the extreme-x goalkeeper proxy. The index says which line is closest to the projected position, for the carrier's own team and for the opposition. Opponent indices follow the carrier's attacking axis, whereas own-team indices keep the opposite-axis ordering used in pass selection.
- **Signed distance to the nearest own-team line** and **to the nearest opposition line.** How far that line is from the projected position along the attacking direction. A positive value means the line is ahead of the point and a negative value means it is behind. A small positive distance to the opposition line means the carrier is about to meet it.

The other two features use the carrier's *current* position.

- **Teammates ahead** and **opponents ahead.** How many players of each team are further up the pitch than the carrier, along the attacking direction. They show how many teammates are in front (potential forward options) and how many opponents are still to be beaten. For a set of players $G$ and a point $r$,

  $$
  N_G(r)=\sum_{k\in G}\mathbf 1\{\delta\,x_k>\delta\,r_x\},\qquad \delta\in\{-1,+1\},
  $$

  where $\delta$ is the carrier team's attacking direction for both rosters. The carrier is excluded and tracked goalkeepers are kept. The count is taken over the whole pitch width.

### Passing opportunities (11 features)

These features summarize what a pass would look like from the current state. A pass has two uncertain parts: *where* it would be directed and *whether it would succeed*. Two separately fitted PRESS components provide one probability for each part, for every candidate destination $j$ on the native one-metre pitch grid:

$$
\pi_j(s)=P(J=j\mid A=\mathrm{pass},s),\qquad
p_j(s)=P(\mathrm{completion}\mid J=j,A=\mathrm{pass},s).
$$

The first, $\pi_j$, says how likely a passer in this situation is to target cell $j$ and comes from the [pass-destination model](../03_pass_selection/03_pass_selection.ipynb). The second, $p_j$, says how likely a pass to that cell is to arrive and comes from the [pass-probability model](../02_pass_probability/02_pass_probability.ipynb). The action model uses the flat destination model with its fitted validation temperature and isotonic-calibrated completion probabilities. Both are evaluated from the current game state, without the destination or outcome of the actual future action.

Multiplying them splits every possible pass into two branches, a completed pass and a failed pass:

$$
w_j^+=\pi_jp_j,\qquad w_j^-=\pi_j(1-p_j).
$$

The weight $w_j^+$ is the chance that the pass goes to cell $j$ and succeeds, and $w_j^-$ is the chance that it goes there and fails.

- **Expected pass success.** The overall chance that a pass would succeed, averaged over the likely destinations. A high value means passing is easy right now.

  $$
  P(\mathrm{completion}\mid A=\mathrm{pass},s)=\sum_j\pi_jp_j=\sum_jw_j^+.
  $$

The other ten features take a spatial property $\Phi(j)$ of each destination, defined for the carrier's team and for the opponent, and add it up separately over the two branches:

$$
\Phi^+(s)=\sum_jw_j^+\Phi(j),\qquad
\Phi^-(s)=\sum_jw_j^-\Phi(j).
$$

The branch determines which team holds the ball after the pass. With $w_j^+$ the carrier's team keeps possession, and $\Phi^+$ is the expected value of $\Phi$ at the place where the pass would be received. With $w_j^-$ the opponent recovers the ball near the destination, and $\Phi^-$ is the expected value of $\Phi$ at the place where it would recover the ball. Line indices and counts keep the carrier team's reference in both branches: the indices identify the nearest line of each team and the counts use the carrier team's attacking direction for both rosters, so they describe the same spatial configuration whichever team holds the ball, and in the failure branch it is the configuration around the place where the opponent recovers. Expected threat measures the value of possession, so it is evaluated for the team holding the ball and oriented towards the goal that team attacks: $X_{own}$ with $w_j^+$ and $X_{opp}$ with $w_j^-$. $\Phi^+$ and $\Phi^-$ are *joint* weighted sums, without division by the overall probability of success or failure, so a branch that is unlikely contributes a small number whatever its destinations look like. Failure is represented at the destination assigned by the destination-choice model, independently of the observed interception or recovery point of the current pass.

- **Expected threat (2 features).** Expected threat is taken for the team that holds the ball in each branch. $\sum_jw_j^+X_{own}(j)$ is the expected threat of the carrier's team after a completed pass: how much a successful pass would advance the attack. $\sum_jw_j^-X_{opp}(j)$ is the expected threat of the opponent after recovering the ball from a failed pass: how dangerous it would be to lose the ball there.
- **Pressure-line structure (4 features).** The nearest own-team and opponent line indices, as defined in the previous group but evaluated at each destination from the current player positions, are aggregated once with $w_j^+$ and once with $w_j^-$:

  $$
  \sum_jw_j^+L_{own}(j),\quad \sum_jw_j^-L_{own}(j),\quad
  \sum_jw_j^+L_{opp}(j),\quad \sum_jw_j^-L_{opp}(j).
  $$

  With $w_j^+$ the indices describe the lines of the two teams near the point where the pass would be received, and with $w_j^-$ near the point where the opponent would recover the ball.
- **Player positioning (4 features).** How many teammates and opponents would still be ahead of the likely destinations. A backward destination can have more players ahead of it than a forward one. The ahead-count function $N_G$ is evaluated at every candidate destination for teammates and opponents, with the carrier team's attacking direction for both rosters, and aggregated the same way:

  $$
  \sum_jw_j^+N_{own}(j),\quad \sum_jw_j^-N_{own}(j),\quad
  \sum_jw_j^+N_{opp}(j),\quad \sum_jw_j^-N_{opp}(j).
  $$

  With $w_j^+$, $N_{own}$ is the expected number of teammates ahead of the receiver, the forward options of the carrier's team, and $N_{opp}$ is the expected number of opponents still to be beaten. With $w_j^-$ the counted players are those behind the ball line in the attacking direction of the team that recovers the ball: $N_{opp}$ counts the teammates of the recovering team and $N_{own}$ its opponents, with the original carrier already excluded. Each team fields a fixed number of players, so counting in the opponent's direction gives the complementary number and the two conventions carry the same information.

Together, the eleven features give the classifier the likelihood, location and consequences of passing, without choosing a destination itself.

### Possession and match context (5 features)

These features describe how the current individual control began and the match situation.

- **Start type** (categorical). The event that gave the player the ball: a reception, interception, recovery or restart.
- **Previous action** (categorical). What the player did immediately before. It keeps the start context until the player completes a full one-second carry, and from then on it records `ball_drive`.
- **Match minute.** Time elapsed in the match.
- **Score difference.** Carrier-team goals minus opponent goals at that frame.
- **Completed carries.** The number of complete one-second carry intervals already performed during the current individual possession, in other words how long the player has already been on the ball. It is a history summary available at decision time. A pass released after 0.4 s has zero completed carries and keeps its start context; a pass released after 2.4 s has two completed carries and `ball_drive` as previous action.

The two categorical variables keep their original labels, including `unknown`, and the encoded vocabulary is fitted on training data. The [possession and checkpoint vignette](../setup/possession_and_checkpoint_setup.ipynb) describes the underlying segmentation.

## 3. Likelihood and learning {#likelihood-and-learning}

The model is trained on observed decisions. Each decision $i$ has two ingredients: the feature vector $z_i$ described above, which summarizes the game state at that moment, and its outcome $y_i$, equal to 1 if the player passed and 0 if the player carried.

XGBoost learns the probability of passing from these pairs. It builds an additive ensemble of regression trees $f_\psi$, in which each new tree corrects the errors left by the previous ones. The ensemble returns a score for every decision, and a logistic function turns that score into the probability of passing:

$$
q_i=P(y_i=1\mid z_i)=\frac{1}{1+\exp[-f_\psi(z_i)]}.
$$

Training chooses the parameters $\psi$ that minimize the regularized Bernoulli negative log-likelihood, which penalizes confident predictions that disagree with the observed action:

$$
\mathcal L(\psi)=-\sum_{i=1}^{n}\left[y_i\log q_i+(1-y_i)\log(1-q_i)\right]+\Omega(\psi),
$$

where $\Omega(\psi)$ penalizes tree complexity and leaf weights to limit overfitting. The [action-selection benchmark](../../benchmarks/04_action_selection.qmd) reports the hyperparameter search, match-wise validation and predictive performance.

## 4. Feature importance and interpretation {#feature-importance-and-interpretation}

The contribution of each feature to the fitted model is measured with XGBoost split gain. The figure reports the total gain attributed to each conceptual feature, with one-hot encoded levels grouped under their original categorical input. These values describe how the trained ensemble uses its inputs.

![Feature importance for the fitted action-selection model.](../../figures/feature_importance_action_selection.png)

The carry-success probability $1-d(s)$ (`drive_success_1s`) is the most influential feature: it accounts for **22.09%** of total gain in the production model and ranks first in all five evaluation folds. Carrier speed follows at **8.00%**, so the player's immediate capacity to advance with the ball is central to the pass–carry decision. The passing-option summaries also contribute substantially: the own-team line index weighted by successful passing outcomes (`team_line_pass_success`) accounts for **6.85%**, and the opposition line index weighted by failed outcomes (`opp_line_pass_failure`) for **5.32%**. BCOS contributes **4.10%** and the previous-action context **4.04%**.

Together, these results are consistent with a classifier that combines the feasibility of continuing the carry with the spatial consequences of the available passing opportunities. Correlated features can share gain among themselves, and split gain depends on the opportunities a feature offers for tree partitioning. The full fold-wise analysis is documented in the [feature-importance analysis](../../data/benchmarks/feature_importance/README.md).

## 5. Application to a game state {#application-to-a-game-state}

Brisbane Roar FC–Perth Glory: N. Acosta chooses a pass at checkpoint 54295, and J. Hingert receives it at frame 54309. The first plot reports the model's probability of choosing a pass at that checkpoint. Set `SKILLCORNER_OPENDATA` to your local open-data repository ([Data setup](../../data.qmd)).

Context-label meanings and the `previous_action` convention are defined in the [User guide](../setup/possession_and_checkpoint_setup.ipynb#context-labels).


In [3]:
from pathlib import Path
import os
import pandas as pd
import matplotlib.pyplot as plt
import jax

jax.config.update("jax_enable_x64", True)
from PRESS import PRESS

opendata = Path(os.environ.get("SKILLCORNER_OPENDATA", Path.cwd().parent / "opendata"))
match_dir = opendata / "data/processed/press/matches/1925299"
press = PRESS.from_skillcorner(
    match_dir, actions_path=match_dir / "1925299_action_selection_base.parquet"
)
frame_id = 54295

# Observed context: getters resolve the frame and possession annotations.
frame = press.get_frame(frame_id).iloc[0]
context = press.get_frame_context(frame_id)
match_clock_min = pd.to_timedelta(frame.timestamp).total_seconds() / 60

# Score is absent from these tracking rows: read the current possession's
# dynamical event rather than inserting a constant in the model call.
possessions = press.get_player_possessions()
current_possession = possessions.loc[
    possessions.frame_start.le(frame_id) & possessions.frame_end.ge(frame_id)
].iloc[0]
events = pd.read_csv(
    opendata / "data/matches" / match_dir.name / f"{match_dir.name}_dynamic_events.csv",
    low_memory=False,
)
origin = events.loc[
    events.event_type.eq("player_possession")
    & events.event_id.eq(current_possession.possession_id)
].iloc[0]
score_difference = float(origin.team_score - origin.opponent_team_score)

from PRESS.model import PassSelectionModel
from PRESS.extras import BCOS

xT_grid = PassSelectionModel.pretrained().xt_grid
area = float(BCOS(press.tracking_data).compute_bcos(frame_id)["BCOS area"])
result = press.compute_frame(
    frame_id, xT_grid, bcos_value=area,
    start_type=context["start_type"], previous_action=context["previous_action"],
    completed_canonical_drives=context["completed_canonical_drives"],
    match_clock_min=match_clock_min, score_difference_prefix=score_difference,
)

from PRESS.visualization import plot_action_selection

plot_action_selection(press, frame_id)
plt.savefig("figures/vignette_04_action_pitch.png", dpi=170)
plt.savefig("figures/vignette_04_action_pitch.svg")
plt.close()


![The real decision checkpoint, with predicted P(pass) above the pitch.](../../figures/vignette_04_action_pitch.png)

In [5]:
from PRESS.visualization import plot_frame

reception_frame = 54309
plot_frame(press, reception_frame, title="Matched reception: J. Hingert · frame 54309")
plt.savefig("figures/vignette_04_action_reception.png", dpi=170)
plt.savefig("figures/vignette_04_action_reception.svg")
plt.close()


![The matched reception of the same pass.](../../figures/vignette_04_action_reception.png)

The [benchmark](../../benchmarks/04_action_selection.qmd) contains the cohort definition, match-disjoint validation, XGBoost tuning, held-out metrics and published comparison. The observed example illustrates the fitted model.
